In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("30-statistical-eda")
dfs   = register_views(spark)
emp   = dfs["employees"]
dept  = dfs["departments"]
sal   = dfs["salary_history"]
pr    = dfs["performance_reviews"]
lr    = dfs["leave_requests"]


# ── Section 1 ── describe() on employees ──────────────────────────────────────
# count/mean/stddev/min/max for salary
# NOTE: Spark describe() EXCLUDES NULL values from count and all aggregates
#       → count=39 (not 41) because emp 10 and emp 15 have NULL salary
#       → emp 19 salary=0.0 IS included and pulls min to 0.0
print("\n── Section 1: describe() on employees ──")
emp.describe("salary").show()
# Spark UI → SQL tab: this triggers a single-stage plan; no shuffle needed
print("NOTE: count in describe() excludes NULLs → expect 39, not 41")
print(f"Actual rows: {emp.count()}, Non-null salary rows: {emp.filter(F.col('salary').isNotNull()).count()}")


# ── Section 2 ── Percentiles via approxQuantile ───────────────────────────────
# approxQuantile(col, quantiles, relativeError)
# relativeError=0.01 → within 1% of exact value; 0.0 = exact (expensive on large data)
# NOTE: emp 19 salary=0.0 pulls p25 down — removing it changes the quartile picture
print("\n── Section 2: Percentiles ──")

quantiles = [0.25, 0.50, 0.75, 0.90, 0.95]
# approxQuantile ignores NULLs automatically
pcts = emp.approxQuantile("salary", quantiles, 0.01)
labels = ["p25", "p50 (median)", "p75", "p90", "p95"]
print("Salary percentiles (with emp 19 salary=0.0 included):")
for label, val in zip(labels, pcts):
    print(f"  {label:<15}: {val:>12,.2f}")

# Re-run excluding salary=0.0 to show emp 19's effect
pcts_no_zero = emp.filter(F.col("salary") > 0).approxQuantile("salary", quantiles, 0.01)
print("\nSalary percentiles (emp 19 salary=0.0 excluded):")
for label, val in zip(labels, pcts_no_zero):
    print(f"  {label:<15}: {val:>12,.2f}")
print("Observation: p25 rises noticeably when the 0.0 outlier is removed")


# ── Section 3 ── IQR and outlier bounds ───────────────────────────────────────
# IQR = p75 - p25; lower_fence = p25 - 1.5*IQR; upper_fence = p75 + 1.5*IQR
# Employees outside the fence are flagged as statistical outliers
print("\n── Section 3: IQR and outlier detection ──")

p25, p75 = pcts[0], pcts[2]
iqr         = p75 - p25
lower_fence = p25 - 1.5 * iqr
upper_fence = p75 + 1.5 * iqr
print(f"p25={p25:,.2f}  p75={p75:,.2f}  IQR={iqr:,.2f}")
print(f"Lower fence: {lower_fence:,.2f}   Upper fence: {upper_fence:,.2f}")

outliers_iqr = emp.filter(
    F.col("salary").isNotNull() &
    ((F.col("salary") < lower_fence) | (F.col("salary") > upper_fence))
)
print("\nOutlier employees (outside 1.5×IQR fence):")
outliers_iqr.select("emp_id", "first_name", "last_name", "dept_id", "salary").show()
# emp 19 (salary=0.0) will appear as a lower outlier if fence > 0


# ── Section 4 ── Frequency distribution (salary buckets) ─────────────────────
# Salary bins: 0-40k, 40k-60k, 60k-80k, 80k-100k, 100k-120k, 120k+
# NULL salaries fall into "unknown" bucket
print("\n── Section 4: Salary frequency distribution ──")

salary_buckets = emp.withColumn(
    "salary_bin",
    F.when(F.col("salary").isNull(),           F.lit("unknown"))
     .when(F.col("salary") == 0.0,             F.lit("0 (zero)"))
     .when(F.col("salary") <  40_000,          F.lit("< 40k"))
     .when(F.col("salary") <  60_000,          F.lit("40k-60k"))
     .when(F.col("salary") <  80_000,          F.lit("60k-80k"))
     .when(F.col("salary") < 100_000,          F.lit("80k-100k"))
     .when(F.col("salary") < 120_000,          F.lit("100k-120k"))
     .otherwise(                                F.lit("120k+"))
)

bucket_counts = salary_buckets.groupBy("salary_bin") \
    .agg(F.count("*").alias("count")) \
    .orderBy(F.col("count").desc())
bucket_counts.show()

# ASCII bar chart
print("Salary distribution (ASCII bar chart):")
rows = bucket_counts.collect()
max_count = max(r["count"] for r in rows)
for row in rows:
    bar = "█" * int(row["count"] / max_count * 30)
    print(f"  {row['salary_bin']:<12} | {bar:<30} {row['count']}")


# ── Section 5 ── corr() — salary vs performance rating ───────────────────────
# Join employees to performance_reviews; compute Pearson correlation
# NOTE: Spark corr() ignores rows where either value is NULL
# Spark UI → SQL tab: look for the HashAggregate + Exchange nodes from the join
print("\n── Section 5: Correlation — salary vs performance rating ──")

emp_pr = emp.join(
    pr.select("emp_id", "rating"),
    on="emp_id",
    how="inner"
).select("emp_id", "salary", "rating")

correlation = emp_pr.stat.corr("salary", "rating")
print(f"Pearson correlation (salary vs rating): {correlation:.4f}")
print("Interpretation: value near 0 → weak linear relationship in this small dataset")
print("NOTE: corr() drops rows where salary or rating is NULL before computing")


# ── Section 6 ── Skewness and Kurtosis ───────────────────────────────────────
# Skewness > 0 → right-skewed (tail extends toward high salaries)
# Kurtosis > 3 → leptokurtic (heavier tails than normal distribution)
# Engineering dept (34% of employees) inflates the upper tail → positive skew expected
print("\n── Section 6: Skewness and Kurtosis ──")

skew_kurt = emp.filter(F.col("salary").isNotNull()) \
               .agg(
                   F.skewness("salary").alias("skewness"),
                   F.kurtosis("salary").alias("kurtosis"),
                   F.avg("salary").alias("mean"),
                   F.stddev("salary").alias("stddev"),
               )
skew_kurt.show()
print("Interpretation:")
print("  skewness > 0 → right tail (Engineering high salaries pull mean above median)")
print("  kurtosis > 3 → leptokurtic (outliers more extreme than normal)")
print("  Note: emp 19 salary=0.0 adds a left-tail outlier that competes with right skew")

# By department: which dept contributes most to right-skew?
emp.join(dept.select("dept_id", "dept_name"), on="dept_id") \
   .groupBy("dept_name") \
   .agg(
       F.skewness("salary").alias("skewness"),
       F.avg("salary").alias("avg_salary"),
       F.count("*").alias("headcount"),
   ).orderBy(F.col("skewness").desc_nulls_last()) \
   .show()


# ── Section 7 ── Coefficient of variation per department ─────────────────────
# CV = stddev / mean — measures relative dispersion regardless of scale
# Higher CV → more salary variance within that department
print("\n── Section 7: Coefficient of variation per department ──")

cv_df = emp.join(dept.select("dept_id", "dept_name"), on="dept_id") \
           .groupBy("dept_id", "dept_name") \
           .agg(
               F.avg("salary").alias("avg_salary"),
               F.stddev("salary").alias("stddev_salary"),
               F.count("*").alias("headcount"),
           ) \
           .withColumn(
               "cv_pct",
               F.round(F.col("stddev_salary") / F.col("avg_salary") * 100, 2)
           ) \
           .orderBy(F.col("cv_pct").desc_nulls_last())

cv_df.select("dept_name", "avg_salary", "stddev_salary", "cv_pct", "headcount").show()
print("Dept with highest CV has most internal salary spread (possible band inconsistency)")


# ── Section 8 ── Hire cohort analysis ─────────────────────────────────────────
# Group by hire year; count employees and compute avg salary per cohort
# Shows whether newer hires earn more/less than tenured employees
print("\n── Section 8: Hire cohort analysis ──")

cohort = emp.withColumn("hire_year", F.year("hire_date")) \
            .groupBy("hire_year") \
            .agg(
                F.count("*").alias("headcount"),
                F.avg("salary").alias("avg_salary"),
                F.min("salary").alias("min_salary"),
                F.max("salary").alias("max_salary"),
            ) \
            .orderBy("hire_year")

cohort.show()
print("NOTE: emp 41 has hire_date=2025-08-01 (future) → appears as 2025 cohort (data flaw)")


# ── Section 9 ── Performance rating distribution ─────────────────────────────
# describe() on rating; count NULLs separately (describe excludes them)
# Histogram bins for rating values (typically 1-5 scale)
print("\n── Section 9: Performance rating distribution ──")

pr.describe("rating").show()

null_ratings = pr.filter(F.col("rating").isNull()).count()
print(f"NULL ratings: {null_ratings} (excluded from describe() count above)")

rating_hist = pr.withColumn(
    "rating_bin",
    F.when(F.col("rating").isNull(),        F.lit("NULL"))
     .when(F.col("rating") < 2,             F.lit("1 (Poor)"))
     .when(F.col("rating") < 3,             F.lit("2 (Below Avg)"))
     .when(F.col("rating") < 4,             F.lit("3 (Average)"))
     .when(F.col("rating") < 5,             F.lit("4 (Good)"))
     .otherwise(                             F.lit("5 (Excellent)"))
).groupBy("rating_bin").agg(F.count("*").alias("count")).orderBy("rating_bin")
rating_hist.show()


# ── Section 10 ── Leave request duration stats ───────────────────────────────
# Duration in days = datediff(end_date, start_date) + 1 (inclusive)
# describe() on duration; identify the longest leave
print("\n── Section 10: Leave request duration stats ──")

lr_dur = lr.withColumn(
    "duration_days",
    F.datediff(F.col("end_date"), F.col("start_date")) + 1
)
lr_dur.describe("duration_days").show()

print("Longest leave requests:")
lr_dur.orderBy(F.col("duration_days").desc()).select(
    "request_id", "emp_id", "leave_type", "start_date", "end_date", "duration_days"
).show(5)

avg_dur = lr_dur.agg(F.avg("duration_days").alias("avg_days"),
                     F.max("duration_days").alias("max_days")).collect()[0]
print(f"Average leave duration: {avg_dur['avg_days']:.1f} days   "
      f"Longest: {avg_dur['max_days']} days")


# ── Section 11 ── Cross-table join for full employee profile ──────────────────
# Join: employees + departments + salary_history
# Derive: current salary (latest salary_after per emp), salary change count
# Spark UI → SQL tab: notice the two BroadcastHashJoin nodes (dept is small → auto-broadcast)
# Storage tab: nothing cached here — all lazy evaluation chains
print("\n── Section 11: Full employee profile ──")

# Latest salary per employee from salary_history
w_latest = Window.partitionBy("emp_id").orderBy(F.col("effective_date").desc(), F.col("hist_id").desc())
sal_latest = sal.withColumn("rn", F.row_number().over(w_latest)) \
                .filter(F.col("rn") == 1) \
                .select(
                    "emp_id",
                    F.col("salary_after").alias("latest_sal_history"),
                    F.col("effective_date").alias("latest_change_date"),
                )

# Change count per employee
sal_changes = sal.groupBy("emp_id") \
                 .agg(F.count("*").alias("salary_change_count"))

# Full profile join
full_profile = emp \
    .join(dept.select("dept_id", "dept_name", "location"), on="dept_id", how="left") \
    .join(sal_latest,   on="emp_id", how="left") \
    .join(sal_changes,  on="emp_id", how="left") \
    .select(
        "emp_id",
        F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("name"),
        "dept_name",
        "location",
        "job_title",
        "salary",
        "latest_sal_history",
        "latest_change_date",
        "salary_change_count",
        "hire_date",
        "status",
    )

print("Full employee profile (sample 10 rows):")
full_profile.show(10, truncate=False)
# Spark UI → SQL tab: open the query plan; click Exchange to see shuffle details
# SQL → Details: look for "Sort Merge Join" or "Broadcast Hash Join" labels


# ── Section 12 ── Write EDA summary stats to CSV ─────────────────────────────
# Department-level aggregates: headcount, avg/min/max salary, stddev, CV
# Spark UI → Jobs tab: .write triggers the action; click to see stage DAG
# Stage count typically: 1 scan + 1 exchange (shuffle for groupBy) + 1 write
print("\n── Section 12: Write dept EDA summary to CSV ──")

dept_summary = emp \
    .join(dept.select("dept_id", "dept_name"), on="dept_id", how="left") \
    .groupBy("dept_id", "dept_name") \
    .agg(
        F.count("*").alias("headcount"),
        F.count("salary").alias("non_null_salary_count"),
        F.avg("salary").alias("avg_salary"),
        F.stddev("salary").alias("stddev_salary"),
        F.min("salary").alias("min_salary"),
        F.max("salary").alias("max_salary"),
        F.sum("salary").alias("total_salary_spend"),
    ) \
    .withColumn("cv_pct", F.round(F.col("stddev_salary") / F.col("avg_salary") * 100, 2)) \
    .withColumn("pct_of_workforce",
        F.round(F.col("headcount") / F.sum("headcount").over(Window.rowsBetween(
            Window.unboundedPreceding, Window.unboundedFollowing)) * 100, 1)) \
    .orderBy(F.col("headcount").desc())

dept_summary.show()
print("NOTE: Engineering shows 34% of workforce — the dept skew is visible in headcount")

out_csv = output_path("csv/eda_dept_summary")
dept_summary.coalesce(1).write.mode("overwrite").option("header", "true").csv(out_csv)
print(f"Dept EDA summary written to: {out_csv}")
# Spark UI → Storage tab: coalesce(1) adds a narrow dependency stage (no full shuffle)
# SQL tab → see the Window + Aggregation + Write plan


stop_and_wait(spark)